# Capítulo 7 · VQE para la Molécula H₂

## Objetivos

1. Comprender cómo se mapea un problema de química cuántica a un Hamiltoniano de qubits.
2. Implementar el algoritmo VQE completo para H₂ (geometría de equilibrio) usando Qiskit Nature.
3. Calcular la curva de energía de disociación comparando HF, FCI exacta y VQE-UCCSD.
4. Analizar el error de correlación electrónica capturado por el ansatz UCCSD.

---

## 7A.1 La molécula H₂ como banco de pruebas cuántico

La molécula de hidrógeno H₂ es el sistema más simple con correlación electrónica. Consta de dos electrones en dos orbitales moleculares (σ y σ*). En la base STO-3G se necesitan **4 qubits** bajo el mapeo Jordan-Wigner, y el ansatz UCCSD tiene solo **3 parámetros variacionales**.

El Hamiltoniano electrónico se expresa como operadores de Pauli mediante el mapeo de Jordan-Wigner:

$$\hat{H} = \sum_k h_k \hat{P}_k, \quad \hat{P}_k \in \{I, X, Y, Z\}^{\otimes n}$$

El valor esperado de la energía para un estado $|\psi(\boldsymbol{\theta})\rangle$ producido por el ansatz UCCSD es:

$$E(\boldsymbol{\theta}) = \langle \psi(\boldsymbol{\theta}) | \hat{H} | \psi(\boldsymbol{\theta}) \rangle$$

y el VQE minimiza esta cantidad optimizando los parámetros $\boldsymbol{\theta}$.

---

## 7A.2 Flujo de trabajo

```
Driver PySCF  →  Hamiltoniano fermiónico  →  Mapeo JW  →  Hamiltoniano de qubits
                                                                     ↓
                                            Ansatz UCCSD (HF + excitaciones)
                                                                     ↓
                                            VQE + SLSQP (optimización clásica)
                                                                     ↓
                                            Energía total = E_elec + E_nucl
```

**Dependencias necesarias:**
```bash
pip install qiskit qiskit-nature qiskit-algorithms pyscf
```

In [1]:
import sys
from typing import List

import numpy as np
import matplotlib.pyplot as plt

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import StatevectorEstimator

print("Entorno:", sys.executable)
print("Módulos cargados correctamente.")

Entorno: c:\Program Files\Python312\python.exe
Módulos cargados correctamente.


## 7A.3 Paso 1 — Especificación molecular

Definimos la geometría de H₂ en la distancia de enlace de equilibrio ($R = 0.7414$ Å) y la base mínima STO-3G.

In [2]:
# ── Geometría de equilibrio ───────────────────────────────────────
BOND_LENGTH = 0.7414      # Å (distancia experimental H-H)
GEOMETRY    = f"H 0 0 0; H 0 0 {BOND_LENGTH}"
BASIS_SET   = "sto3g"     # Base mínima: 2 orbitales espaciales → 4 qubits (JW)
CHARGE      = 0
SPIN        = 0           # Singlete (multiplicidad 1)

print(f"Molécula : H₂")
print(f"Geometría: {GEOMETRY}")
print(f"Base     : {BASIS_SET}")
print(f"Carga    : {CHARGE}")
print(f"Spin     : {SPIN}  →  multiplicidad {SPIN + 1}")

Molécula : H₂
Geometría: H 0 0 0; H 0 0 0.7414
Base     : sto3g
Carga    : 0
Spin     : 0  →  multiplicidad 1


## 7A.4 Paso 2 — Driver PySCF y Hamiltoniano fermiónico

El driver corre un cálculo Hartree-Fock (HF) clásico con PySCF para obtener los integrales de un electrón $h_{pq}$ y dos electrones $h_{pqrs}$ que definen el Hamiltoniano electrónico de segunda cuantización:

$$\hat{H}_{el} = \sum_{pq} h_{pq} a_p^\dagger a_q + \frac{1}{2}\sum_{pqrs} h_{pqrs} a_p^\dagger a_q^\dagger a_s a_r$$

In [3]:
# ── Driver PySCF ──────────────────────────────────────────────────
driver = PySCFDriver(
    atom=GEOMETRY,
    basis=BASIS_SET,
    charge=CHARGE,
    spin=SPIN,
    unit=DistanceUnit.ANGSTROM,
)

driver_result = driver.run()
print("✓ Driver PySCF ejecutado.")

# Metadatos del sistema electrónico
num_particles         = driver_result.num_particles
num_spatial_orbitals  = driver_result.num_spatial_orbitals
nuclear_repulsion     = float(driver_result.nuclear_repulsion_energy)
second_q_hamiltonian  = driver_result.hamiltonian.second_q_op()

print(f"\nPartículas (α, β)   : {num_particles}")
print(f"Orbitales espaciales: {num_spatial_orbitals}")
print(f"Repulsión nuclear   : {nuclear_repulsion:.10f} Ha")

MissingOptionalLibraryError: "The 'pyscf' library is required to use 'PySCFDriver'.  See https://pyscf.org/install.html."

## 7A.5 Paso 3 — Mapeo Jordan-Wigner

El mapeo Jordan-Wigner (JW) convierte los operadores fermiónicos de creación/aniquilación en cadenas de Pauli. Para $n$ orbitales de spin se necesitan $n$ qubits.

Para el modo $k$-ésimo el mapeo es:

$$a_k^\dagger \to \left(\bigotimes_{j<k} Z_j\right) \otimes \frac{X_k - iY_k}{2}, \quad a_k \to \left(\bigotimes_{j<k} Z_j\right) \otimes \frac{X_k + iY_k}{2}$$

In [ ]:
# ── Mapeo Jordan-Wigner ───────────────────────────────────────────
mapper            = JordanWignerMapper()
qubit_hamiltonian = mapper.map(second_q_hamiltonian)

pauli_terms = qubit_hamiltonian.to_list()

print(f"Qubits necesarios: {qubit_hamiltonian.num_qubits}")
print(f"Términos de Pauli: {len(pauli_terms)}")
print("\nPrimeros 8 términos del Hamiltoniano:")
for i, (pauli_str, coeff) in enumerate(pauli_terms[:8], start=1):
    print(f"  {i:2d}. {coeff:+.8f} · {pauli_str}")

## 7A.6 Paso 4 — Ansatz UCCSD

El ansatz **Unitary Coupled Cluster Singles and Doubles (UCCSD)** parte del estado Hartree-Fock (HF) y aplica excitaciones unitarias:

$$|\psi(\boldsymbol{\theta})\rangle = e^{\hat{T}(\boldsymbol{\theta}) - \hat{T}^\dagger(\boldsymbol{\theta})} |\psi_{\text{HF}}\rangle$$

donde $\hat{T} = \hat{T}_1 + \hat{T}_2$ contiene excitaciones simples (single) y dobles (double).

Para H₂ en STO-3G solo hay una excitación doble relevante → **3 parámetros** variacionales.

In [ ]:
# ── Estado inicial Hartree-Fock ───────────────────────────────────
hf_state = HartreeFock(
    num_spatial_orbitals=num_spatial_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
)

# ── Ansatz UCCSD ──────────────────────────────────────────────────
ansatz = UCCSD(
    num_spatial_orbitals=num_spatial_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
    initial_state=hf_state,
)

print(f"Parámetros variacionales: {ansatz.num_parameters}")
print(f"Profundidad del circuito : {ansatz.decompose().depth()}")
print()
print("Circuito UCCSD (nivel alto):")
print(ansatz.draw('text'))

## 7A.7 Paso 5 — Ejecución VQE

El VQE alterna entre:
- **Evaluación cuántica**: preparar $|\psi(\boldsymbol{\theta})\rangle$ y medir $\langle H \rangle$.
- **Optimización clásica**: ajustar $\boldsymbol{\theta}$ con SLSQP hasta minimizar $E(\boldsymbol{\theta})$.

Usamos `StatevectorEstimator` (simulación exacta sin ruido).

In [ ]:
# ── Configuración VQE ─────────────────────────────────────────────
estimator  = StatevectorEstimator()
optimizer  = SLSQP(maxiter=120, eps=1e-6)

energy_history: List[float] = []

def callback(eval_count, params, mean, metadata):
    """Registra la energía en cada evaluación del optimizador."""
    energy_history.append(float(mean))
    if eval_count == 1 or eval_count % 5 == 0:
        print(f"  Eval {eval_count:3d}: E = {float(mean):+.10f} Ha")

vqe = VQE(
    estimator=estimator,
    ansatz=ansatz,
    optimizer=optimizer,
    callback=callback,
)

print("Ejecutando VQE...")
result = vqe.compute_minimum_eigenvalue(qubit_hamiltonian)
print("\n✓ VQE completado.")

In [ ]:
# ── Resultados ─────────────────────────────────────────────────────
vqe_electronic = float(np.real(result.eigenvalue))
vqe_total      = vqe_electronic + nuclear_repulsion

# Referencias H₂ STO-3G en R = 0.7414 Å
HF_REF  = -1.11733   # Ha  (Hartree-Fock)
FCI_REF = -1.17463   # Ha  (Full CI, resultado exacto en la base)

print("=" * 60)
print(f"Energía electrónica VQE   : {vqe_electronic:+.10f} Ha")
print(f"Repulsión nuclear         : {nuclear_repulsion:+.10f} Ha")
print(f"Energía total VQE         : {vqe_total:+.10f} Ha")
print("-" * 60)
print(f"Referencia HF             : {HF_REF:+.5f} Ha")
print(f"Referencia FCI            : {FCI_REF:+.5f} Ha")
print(f"Error vs FCI              : {abs(vqe_total - FCI_REF) * 1000:.3f} mHa")
print(f"Energía de correlación cap.: {abs(vqe_total - HF_REF) / abs(FCI_REF - HF_REF) * 100:.1f}%")
print("=" * 60)
print(f"\nParámetros óptimos: {result.optimal_point}")
print(f"Evaluaciones totales: {len(energy_history)}")

## 7A.8 Visualización — Convergencia del VQE

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(range(1, len(energy_history) + 1), energy_history,
        marker='o', markersize=4, linewidth=1.8,
        color='#58a6ff', label='VQE (UCCSD / STO-3G)')

ax.axhline(FCI_REF, linestyle='--', linewidth=1.5,
           color='#f78166', label=f'FCI ref = {FCI_REF} Ha')
ax.axhline(HF_REF,  linestyle=':', linewidth=1.5,
           color='#a5d6ff', label=f'HF  ref = {HF_REF} Ha')

ax.set_xlabel('Evaluación del optimizador', fontsize=12)
ax.set_ylabel('Energía (Ha)', fontsize=12)
ax.set_title('Convergencia del VQE para H₂ (STO-3G, R = 0.7414 Å)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
ax.tick_params(colors='#8b949e')
ax.xaxis.label.set_color('#8b949e')
ax.yaxis.label.set_color('#8b949e')
ax.title.set_color('#e6edf3')

plt.tight_layout()
plt.savefig('vqe_convergence_h2_notebook.png', dpi=150, facecolor='#0d1117')
plt.show()
print("✓ Gráfico guardado.")

## 7A.9 Curva de disociación H₂

Calculamos la energía total para distintas distancias internucleares $R$ y comparamos HF, FCI exacta (PySCF) y VQE-UCCSD. La diferencia FCI − HF es la **energía de correlación**, que el UCCSD captura completamente en sistemas de 2 electrones.

> **Nota**: este cálculo puede tardar varios minutos dependiendo del número de puntos y la máquina.

In [ ]:
import time

def run_h2_vqe(bond_length: float, basis: str = 'sto3g') -> dict:
    """Calcula HF, FCI y VQE-UCCSD para H₂ a una distancia R dada.

    Parámetros
    ----------
    bond_length : float
        Distancia internuclear en Å.
    basis : str
        Base gaussiana (sto3g, 6-31g, cc-pvdz, ...).

    Devuelve
    --------
    dict con campos R, hf_total, fci_total, vqe_total, error_mHa.
    """
    geom   = f"H 0 0 0; H 0 0 {bond_length}"
    driver = PySCFDriver(atom=geom, basis=basis, charge=0, spin=0,
                         unit=DistanceUnit.ANGSTROM)
    res    = driver.run()

    n_part  = res.num_particles
    n_orb   = res.num_spatial_orbitals
    e_nuc   = float(res.nuclear_repulsion_energy)
    hf_elec = float(res.reference_energy)
    hf_tot  = hf_elec + e_nuc

    # FCI exacta usando PySCF directamente
    from pyscf import gto, scf, fci as pyscf_fci
    mol = gto.M(atom=geom, basis=basis, unit='Angstrom', verbose=0)
    mf  = scf.RHF(mol).run()
    cisolver = pyscf_fci.FCI(mf)
    fci_e, _ = cisolver.kernel()
    fci_tot  = float(fci_e)

    # VQE-UCCSD
    mapper  = JordanWignerMapper()
    q_ham   = mapper.map(res.hamiltonian.second_q_op())
    hf_st   = HartreeFock(n_orb, n_part, mapper)
    ans     = UCCSD(n_orb, n_part, mapper, initial_state=hf_st)
    vqe_run = VQE(StatevectorEstimator(), ans, SLSQP(maxiter=150, eps=1e-6))
    vqe_res = vqe_run.compute_minimum_eigenvalue(q_ham)
    vqe_tot = float(np.real(vqe_res.eigenvalue)) + e_nuc

    return dict(
        R=bond_length,
        hf_total=hf_tot,
        fci_total=fci_tot,
        vqe_total=vqe_tot,
        error_mHa=abs(vqe_tot - fci_tot) * 1000,
    )


# Puntos de la curva: 0.35 Å → 2.50 Å
R_values = np.linspace(0.35, 2.50, 18)
results  = []

for R in R_values:
    t0 = time.time()
    data = run_h2_vqe(R, basis='sto3g')
    elapsed = time.time() - t0
    results.append(data)
    print(f"R={R:.3f} Å  HF={data['hf_total']:.6f}  "
          f"FCI={data['fci_total']:.6f}  "
          f"VQE={data['vqe_total']:.6f}  "
          f"err={data['error_mHa']:.3f} mHa  ({elapsed:.1f}s)")

print("\n✓ Curva de disociación completada.")

In [ ]:
# ── Gráfico de la curva de disociación ───────────────────────────
R_arr   = np.array([d['R']         for d in results])
hf_arr  = np.array([d['hf_total']  for d in results])
fci_arr = np.array([d['fci_total'] for d in results])
vqe_arr = np.array([d['vqe_total'] for d in results])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0d1117')

# ── Panel izquierdo: curvas de energía ───
ax1.plot(R_arr, hf_arr,  '--', color='#a5d6ff', linewidth=1.8, label='HF')
ax1.plot(R_arr, fci_arr, '-',  color='#f78166', linewidth=2.0, label='FCI exacta')
ax1.plot(R_arr, vqe_arr, 'o', color='#58a6ff',  linewidth=1.8,
         markersize=5, linestyle='-', label='VQE-UCCSD')

ax1.axvline(0.7414, color='#888', linestyle=':', alpha=0.7,
            label='$R_{eq}$ = 0.7414 Å')
ax1.set_xlabel('Distancia internuclear R (Å)', fontsize=11)
ax1.set_ylabel('Energía total (Ha)', fontsize=11)
ax1.set_title('Curva de disociación H₂ (STO-3G)', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.25)
ax1.set_facecolor('#161b22')
ax1.tick_params(colors='#8b949e')

# ── Panel derecho: error VQE vs FCI ──────
err_mHa = (vqe_arr - fci_arr) * 1000
ax2.plot(R_arr, np.abs(err_mHa), 'o-', color='#e3b341',
         linewidth=1.8, markersize=5)
ax2.axhline(1.6, linestyle='--', color='#f78166', linewidth=1.2,
            label='Precisión química (1.6 mHa)')
ax2.set_xlabel('Distancia internuclear R (Å)', fontsize=11)
ax2.set_ylabel('|Error VQE vs FCI| (mHa)', fontsize=11)
ax2.set_title('Error de correlación residual', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.25)
ax2.set_facecolor('#161b22')
ax2.tick_params(colors='#8b949e')

for ax in (ax1, ax2):
    ax.xaxis.label.set_color('#8b949e')
    ax.yaxis.label.set_color('#8b949e')
    ax.title.set_color('#e6edf3')

plt.tight_layout()
plt.savefig('vqe_dissociation_h2_notebook.png', dpi=150, facecolor='#0d1117')
plt.show()
print("✓ Gráfico guardado.")

## 7A.10 Discusión de resultados

**Geometría de equilibrio:**  
El UCCSD es exacto para 2 electrones, por lo que el VQE converge al FCI con error $<$ 1 mHa (dentro de la **precisión química**, criterio estándar de 1.6 mHa ≈ 1 kcal/mol).

**Curva de disociación:**  
A distancias largas ($R > 1.5$ Å) el carácter multiconfiguracional del estado aumenta. El UCCSD puede fallar puntualmente si el optimizador SLSQP queda atrapado en un mínimo local, dando errores mayores.

**Escala de recursos:**  

| Base | Orbitales espaciales | Qubits (JW) | Parámetros UCCSD |
|------|---------------------|-------------|------------------|
| STO-3G | 2 | 4 | 3 |
| 6-31G | 4 | 8 | 15 |
| cc-pVDZ | 10 | 20 | ∼150 |

---

## 7A.11 Ejercicios propuestos

1. **Base 6-31G**: repite el cálculo VQE con la base `6-31g` (8 qubits, 15 parámetros). ¿Mejora el resultado en la geometría de equilibrio? ¿Cómo escala el tiempo?

2. **Optimizador COBYLA**: sustituye `SLSQP` por `COBYLA(maxiter=500)`. Compara la convergencia y el número de evaluaciones.

3. **Ansatz alternativo**: usa `RealAmplitudes(num_qubits=4, reps=2)` en lugar de UCCSD. ¿Alcanza la precisión química? ¿Por qué sí o no?

4. **Simulación con ruido**: repite el VQE usando `AerSimulator` con el ruido de un backend real de IBM. Analiza el impacto en la energía obtenida.

5. **Energía de correlación**: define la energía de correlación como $E_c = E_{\text{FCI}} - E_{\text{HF}}$. ¿Qué fracción captura el UCCSD a distintas distancias?